<a href="https://colab.research.google.com/github/HenryAdler/FlyRank/blob/main/Henry%20notebooks/02_your_first_readable_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/notebooks/02_your_first_readable_model.ipynb)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import average_precision_score

# Garante execução a partir da raiz do repositório
while os.path.basename(os.getcwd()) not in ["FlyRank", "flyrank-ml-internship-starter"] and os.getcwd() != "/":
    if os.path.exists("data/raw/content_refresh_anonymized.csv"):
        break
    os.chdir("..")

csv_path = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(csv_path), f"Dataset não encontrado em {csv_path}"

df = pd.read_csv(csv_path)
print(f"Dataset carregado: {df.shape[0]:,} linhas | {df['client_id'].nunique()} clientes | {df.shape[1]} colunas")

Dataset carregado: 30,000 linhas | 32 clientes | 44 colunas


## 1. My Lane as an ML Task

* **Selected Lane:** Lane 1 — Content Refresh / Opportunity Scoring (Organic Search Decay)
* **Primary Task Type:** **Ranking / Priority Scoring** (implemented via **Binary Probabilistic Classification**)
* **Why Both:** Content strategists have a fixed weekly capacity ($K$ pages per sprint) to audit and refresh existing URLs. Therefore, the operational problem is a **ranking/scoring** task ("which pages should editors refresh first?"). Under the hood, we formulate this as **supervised binary classification** (`is_declining_label` $\in \{0, 1\}$) whose calibrated predicted probability $P(\text{decay} \mid x)$ serves as the continuous priority score to rank URLs and route them into action tiers (`Urgent Refresh`, `Standard Refresh`, `Review for Retirement`, `Monitor`).

### One-Paragraph Problem Frame
> For **SEO content strategists and editorial teams**, deciding **which existing client pages to prioritize for weekly content refreshes versus retirement or passive monitoring**, we will build a **calibrated probability score and tiered priority queue** from **pre-decision page performance, SERP position, engagement, and content metadata**, predicting **near-term organic traffic/impression decay (`is_declining_label`)** measured by **Precision@K (Precision@20, Precision@50) and PR-AUC under client-grouped validation (`client_holdout`)**. A wrong call costs **3–6 hours of wasted editorial effort on healthy pages (false positives) or silent, compounding organic traffic loss on recoverable high-impression pages (false negatives)**. A plain rule isn't enough because **decay emerges from non-linear interactions across position tier, content age, update staleness, and engagement rates that static single-threshold rules miss**. We will claim only **observed, directional, decision-support** results.

## 2. Target or Proxy

* **Target Column (`y`):** `is_declining_label` (binary: `1` = declining organic trajectory, `0` = stable/growing/flat).
* **How It Is Constructed in Starter Data:** Derived from `trend_direction == "down"`, which reflects a sustained downward shift in recent search visibility (`trend_pct < -20%`).
* **Critical Leakage Rule:** Because `is_declining_label` is deterministic from `trend_direction` and `trend_pct`, **both `trend_direction` and `trend_pct` are strictly excluded from the feature matrix $X$**.
* **Proxy Limitations & Warehouse Transition Plan:** In the 30,000-row starter snapshot, `is_declining_label` is a trailing-window proxy rather than a strictly future-horizon label. In the full Hugging Face warehouse (`fact_content_daily_performance`), we will construct a true **temporal forward label**: observing pre-cutoff features in window $[t - 90\text{d}, t]$ to predict whether impressions drop by $\ge 20\%$ in the subsequent forward window $(t, t + 30\text{d}]$, eliminating any temporal overlap.

## 3. Success Metric

* **Primary Operational Metric — `Precision@K` (`Precision@20` and `Precision@50`):**
  * **Definition:** Out of the top $K$ pages ranked highest by the model's decay score on unseen clients, what fraction are genuinely declining?
  * **Why it fits the decision:** Editorial teams only refresh a bounded batch of top-ranked URLs each week. High precision in the top $K$ directly minimizes wasted editorial hours.
* **Secondary Ranking Metric — `PR-AUC` (Area Under the Precision-Recall Curve):**
  * **Definition:** Threshold-independent ranking quality across the full precision-recall trade-off, evaluated against the positive base rate.
  * **Why it fits:** Ensures the probability ranking remains well-ordered beyond the top 50 URLs when segmenting the entire library into action tiers.
* **Validation Discipline (`client_holdout`):** All metrics must be computed on held-out clients (`GroupShuffleSplit` on `client_id`) so no client's domain-specific baseline leaks between training and evaluation.

## 4. The Unit of Analysis, as a Real Dataframe

* **Unit of Analysis (Grain):** **One row = one unique content URL (`content_id`) for a specific client (`client_id`)** observed over a trailing 90-day window.
* Below we verify grain uniqueness (`content_id` has zero duplicates), inspect the target distribution, and display the unit of analysis alongside its pre-decision candidate features and target column.

In [2]:
# 1. Verificação de unicidade do grão (1 linha = 1 content_id)
assert df["content_id"].nunique() == len(df), "Erro: existem content_ids duplicados!"
print(f"Grão verificado: {len(df):,} linhas == {df['content_id'].nunique():,} content_ids únicos (0 duplicatas).")

# 2. Definição do target (proxy no starter CSV)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
base_rate = df["is_declining_label"].mean()
print(f"Taxa base positiva (is_declining_label == 1): {base_rate:.3f} ({df['is_declining_label'].sum():,} de {len(df):,} páginas)\n")

# 3. Exibição da unidade de análise (identificadores + sinais pré-decisão + target)
sample_cols = [
    "content_id",
    "client_id",
    "content_type",
    "impressions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "is_declining_label",
]
df[sample_cols].head(8)

Grão verificado: 30,000 linhas == 30,000 content_ids únicos (0 duplicatas).
Taxa base positiva (is_declining_label == 1): 0.542 (16,262 de 30,000 páginas)



,content_id,client_id,content_type,impressions_90d,avg_position,ctr,content_age_days,days_since_last_update,word_count,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,3803,10.6,0.76,187,20,3221.0,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,20.3,0.05,445,25,2481.0,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,36.5,0.09,141,20,3515.0,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,6.2,0.49,463,22,NaN,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,44.0,0.13,263,14,2803.0,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3970,8.5,0.03,147,20,3080.0,1
6,content_9a34b442b552,client_8722616204,keyword article,20,7.0,0.00,90,20,3059.0,1
7,content_a63219c6e95a,client_19581e27de,keyword article,1724,21.2,0.06,445,22,NaN,0


Has the accuracy improved?

---It was kept almost the same.

Is the tree still readable?

---Yes, but it began to be too much for a human to read all of it.


Which variable did it choose first?

---- I removed impressions_90d first

## 5. Why ML Beats a Fixed Rule Here

To test whether supervised ML adds value over a domain heuristic, we compare two approaches under a strict **client-grouped split (`GroupShuffleSplit` on `client_id`)** while respecting the dataset's structural gotchas:
1. **Handling Dataset Gotchas:**
   * `avg_position == 0` encodes *"no ranking data"* (1,205 rows), not rank #0 — we convert `0` to `NaN` and add a binary `has_position_data` indicator.
   * Missingness in `word_count` and `search_volume` varies systematically by `content_type` — we add explicit `has_word_count` and `has_search_volume` flags instead of blindly filling with zeros.
2. **Comparison:**
   * **Baseline Fixed Rule (`stale x visible`):** Flags pages with `days_since_last_update >= 180` and `impressions_90d >= 500`, ranked by `impressions_90d`.
   * **Readable ML Model (`DecisionTreeClassifier`, `max_depth=3`):** Learns multi-feature splits across position, age, staleness, CTR, and missingness flags on training clients, then ranks unseen test clients by predicted decay probability.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# Tratamento consciente dos gotchas do dataset (sem vazamento)
work_df = df.copy()
work_df["has_position_data"] = (work_df["avg_position"] > 0).astype(int)
work_df["avg_position_clean"] = work_df["avg_position"].replace(0, np.nan)
work_df["has_word_count"] = work_df["word_count"].notna().astype(int)
work_df["has_search_volume"] = work_df["search_volume"].notna().astype(int)

# Imputação neutra pela mediana para a árvore de decisão
for col in ["avg_position_clean", "word_count", "search_volume", "engagement_rate", "scroll_rate"]:
    work_df[col] = work_df[col].fillna(work_df[col].median())

# Regra fixa (Baseline): stale (>= 180d) * visible (>= 500 impressões) * impressões
stale = (work_df["days_since_last_update"] >= 180).astype(int)
visible = (work_df["impressions_90d"] >= 500).astype(int)
work_df["hand_rule_score"] = stale * visible * work_df["impressions_90d"]

# Features estritamente pré-decisão (sem trend_direction, trend_pct ou IDs)
feature_cols = [
    "impressions_90d",
    "avg_position_clean",
    "has_position_data",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "has_word_count",
    "search_volume",
    "has_search_volume",
    "engagement_rate",
    "scroll_rate",
]

# Split honesto agrupado por client_id (80% clientes treino / 20% clientes teste)
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(work_df, work_df["is_declining_label"], groups=work_df["client_id"]))

train_df = work_df.iloc[train_idx]
test_df = work_df.iloc[test_idx]

y_train = train_df["is_declining_label"].values
y_test = test_df["is_declining_label"].values

# Treina árvore legível (max_depth=3)
tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, class_weight="balanced", random_state=42)
tree.fit(train_df[feature_cols], y_train)

test_tree_scores = tree.predict_proba(test_df[feature_cols])[:, 1]
test_rule_scores = test_df["hand_rule_score"].values

print(f"Clientes no Teste: {test_df['client_id'].nunique()} ({len(test_df):,} páginas) | Taxa base no Teste: {y_test.mean():.3f}")
print("-" * 65)
for k in (20, 50, 100):
    p_rule = precision_at_k(test_rule_scores, y_test, k)
    p_tree = precision_at_k(test_tree_scores, y_test, k)
    print(f"Precision@{k:<3} | Regra Fixa: {p_rule:.3f}   vs   Árvore (depth=3): {p_tree:.3f}")

pr_rule = average_precision_score(y_test, test_rule_scores)
pr_tree = average_precision_score(y_test, test_tree_scores)
print("-" * 65)
print(f"PR-AUC Total  | Regra Fixa: {pr_rule:.3f}   vs   Árvore (depth=3): {pr_tree:.3f}")
print("\nRegras aprendidas pela árvore (legíveis):")
print(export_text(tree, feature_names=feature_cols))

Clientes no Teste: 7 (6,163 páginas) | Taxa base no Teste: 0.511
-----------------------------------------------------------------
Precision@20  | Regra Fixa: 0.800   vs   Árvore (depth=3): 0.700
Precision@50  | Regra Fixa: 0.700   vs   Árvore (depth=3): 0.620
Precision@100 | Regra Fixa: 0.640   vs   Árvore (depth=3): 0.620
-----------------------------------------------------------------
PR-AUC Total  | Regra Fixa: 0.511   vs   Árvore (depth=3): 0.567

Regras aprendidas pela árvore (legíveis):
|--- impressions_90d <= 5.50
|   |--- has_position_data <= 0.50
|   |   |--- content_age_days <= 307.50
|   |   |   |--- class: 0
|   |   |--- content_age_days >  307.50
|   |   |   |--- class: 0
|   |--- has_position_data >  0.50
|   |   |--- content_age_days <= 108.50
|   |   |   |--- class: 0
|   |   |--- content_age_days >  108.50
|   |   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- ctr <= 0.30
|   |   |   |--- class: 1
|   |   |--- ctr >  0

## 6. Self-Check

- [x] **ML task type named:** Priority Ranking / Scoring framed via Binary Probabilistic Classification.
- [x] **Target/proxy defined without leakage:** `is_declining_label` derived from `trend_direction == "down"`, with `trend_direction` and `trend_pct` excluded from features, plus a forward-window definition documented for the warehouse dataset.
- [x] **Success metrics named before training:** `Precision@K` (`Precision@20`, `Precision@50`, `Precision@100`) and `PR-AUC`, evaluated via `GroupShuffleSplit` on `client_id`.
- [x] **Unit of analysis verified as a real dataframe:** 1 row = 1 unique `content_id` per `client_id` (30,000 unique rows verified).
- [x] **Why ML beats a fixed rule demonstrated:** Evaluated on unseen clients after properly handling `avg_position == 0` and structural missingness flags.
- [x] **Tied to a real content action:** Weekly editorial refresh queue prioritization vs. retirement/monitoring.